<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        Baseline Models for Mortality Prediction
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To establish baseline performance for multiple machine learning models in predicting in-hospital mortality at the encounter level, using the preprocessed and leakage-safe dataset without applying any class imbalance handling techniques.

### **Model Setup and Library Imports**

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

xgb_available = True
lgbm_available = True

try:
    from xgboost import XGBClassifier
except ImportError:
    xgb_available = False

try:
    from lightgbm import LGBMClassifier
except ImportError:
    lgbm_available = False

from imblearn.combine import SMOTETomek

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

from pathlib import Path
import joblib

import json

### **Load Split Files**

In [3]:
RAW_BASE_PATH = "../data/processed/splits"
IMPUTED_BASE_PATH = "../data/processed/splits_imputed"

# Raw splits (for models that can handle missing values)
X_train_raw = pd.read_csv(f"{RAW_BASE_PATH}/X_train.csv")
y_train_raw = pd.read_csv(f"{RAW_BASE_PATH}/y_train.csv").squeeze("columns")

X_test_raw = pd.read_csv(f"{RAW_BASE_PATH}/X_test.csv")
y_test_raw = pd.read_csv(f"{RAW_BASE_PATH}/y_test.csv").squeeze("columns")

X_deployment_raw = pd.read_csv(f"{RAW_BASE_PATH}/X_deployment.csv")
y_deployment_raw = pd.read_csv(f"{RAW_BASE_PATH}/y_deployment.csv").squeeze("columns")

# Imputed splits (for models that cannot handle missing values)
X_train_imputed = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_train.csv")
y_train_imputed = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_train.csv").squeeze("columns")

X_test_imputed = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_test.csv")
y_test_imputed = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_test.csv").squeeze("columns")

X_deployment_imputed = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_deployment.csv")
y_deployment_imputed = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_deployment.csv").squeeze("columns")

### **Data validation and Sanity Checks**

In [4]:
assert len(X_train_raw) == len(y_train_raw), "Raw train X/y row count mismatch."
assert len(X_test_raw) == len(y_test_raw), "Raw test X/y row count mismatch."
assert len(X_deployment_raw) == len(y_deployment_raw), "Raw deployment X/y row count mismatch."

assert len(X_train_imputed) == len(y_train_imputed), "Imputed train X/y row count mismatch."
assert len(X_test_imputed) == len(y_test_imputed), "Imputed test X/y row count mismatch."
assert len(X_deployment_imputed) == len(y_deployment_imputed), "Imputed deployment X/y row count mismatch."

for df_name, df in {
    "X_train_raw": X_train_raw,
    "X_test_raw": X_test_raw,
    "X_deployment_raw": X_deployment_raw,
}.items():
    if "_original_order" in df.columns:
        df.drop(columns=["_original_order"], inplace=True)
        print(f"Dropped '_original_order' from {df_name}")

if X_train_imputed.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_train_imputed")
if X_test_imputed.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_test_imputed")
if X_deployment_imputed.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_deployment_imputed")

raw_only_cols = sorted(set(X_train_raw.columns) - set(X_train_imputed.columns))
imputed_only_cols = sorted(set(X_train_imputed.columns) - set(X_train_raw.columns))

print("Raw shapes:")
print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("X_deployment_raw:", X_deployment_raw.shape)

print("\nImputed shapes:")
print("X_train_imputed:", X_train_imputed.shape)
print("X_test_imputed:", X_test_imputed.shape)
print("X_deployment_imputed:", X_deployment_imputed.shape)

print("\nTarget distribution (raw labels):")
print("Train:")
print(y_train_raw.value_counts(normalize=True).rename("proportion"))
print("\nTest:")
print(y_test_raw.value_counts(normalize=True).rename("proportion"))
print("\nDeployment:")
print(y_deployment_raw.value_counts(normalize=True).rename("proportion"))

print("\nColumn differences:")
print("Only in raw:", raw_only_cols)
print("Only in imputed:", imputed_only_cols)

print("\nAll checks passed.")

Dropped '_original_order' from X_train_raw
Dropped '_original_order' from X_test_raw
Dropped '_original_order' from X_deployment_raw
Raw shapes:
X_train_raw: (380461, 49)
X_test_raw: (83356, 49)
X_deployment_raw: (82211, 49)

Imputed shapes:
X_train_imputed: (380461, 49)
X_test_imputed: (83356, 49)
X_deployment_imputed: (82211, 49)

Target distribution (raw labels):
Train:
target
0    0.978289
1    0.021711
Name: proportion, dtype: float64

Test:
target
0    0.978766
1    0.021234
Name: proportion, dtype: float64

Deployment:
target
0    0.978458
1    0.021542
Name: proportion, dtype: float64

Column differences:
Only in raw: []
Only in imputed: []

All checks passed.


### **Helper Definitions**

In [5]:
def compute_metrics(model_name, y_true, y_pred, y_score):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "predicted_positives": int((y_pred == 1).sum()),
    }

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return model.decision_function(X)
    else:
        return model.predict(X)

def run_train_test_experiment(model_configs: dict):
    results = []
    conf_matrices = {}

    for model_name, config in model_configs.items():
        print(f"\nTraining {model_name} ...")

        model = config["model"]
        X_train = config["X_train"]
        y_train = config["y_train"]
        X_test = config["X_test"]
        y_test = config["y_test"]
        native_missing = config["native_missing"]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_score = get_scores(model, X_test)

        result = compute_metrics(model_name, y_test, y_pred, y_score)
        result["native_missing"] = native_missing

        results.append(result)
        conf_matrices[model_name] = confusion_matrix(y_test, y_pred)

    results_df = pd.DataFrame(results).sort_values(
        by=["pr_auc", "recall", "f1"],
        ascending=False
    ).reset_index(drop=True)

    return results_df, conf_matrices

### **Baseline Models**

In [6]:
models_baseline = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=2000, random_state=42),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "RandomForest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "KNN": {
        "model": KNeighborsClassifier(n_neighbors=5),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
}

if xgb_available:
    models_baseline["XGBoost"] = {
        "model": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),
        "X_train": X_train_raw,
        "y_train": y_train_raw,
        "X_test": X_test_raw,
        "y_test": y_test_raw,
        "native_missing": True,
    }

if lgbm_available:
    models_baseline["LightGBM"] = {
        "model": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        "X_train": X_train_raw,
        "y_train": y_train_raw,
        "X_test": X_test_raw,
        "y_test": y_test_raw,
        "native_missing": True,
    }

### **Baseline Models Training and Evaluation (Unbalanced Data)**

In [58]:
baseline_results_df, baseline_conf_matrices = run_train_test_experiment(models_baseline)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("\n=== FINAL TEST RESULTS ===")
display(
    baseline_results_df[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "native_missing",
        ]
    ]
)

print("\n=== TEST CONFUSION MATRICES ===")
for model_name, cm in baseline_conf_matrices.items():
    print(f"\n{model_name}")
    print(cm)


Training LogisticRegression ...

Training DecisionTree ...

Training RandomForest ...

Training GradientBoosting ...

Training KNN ...

Training XGBoost ...

Training LightGBM ...

=== FINAL TEST RESULTS ===


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,XGBoost,0.978826,0.517986,0.040678,0.075432,0.867742,0.174725,139,True
1,LightGBM,0.978226,0.380952,0.040678,0.073507,0.861630,0.154859,189,True
2,GradientBoosting,0.978694,0.453125,0.016384,0.031625,0.857717,0.145876,64,False
3,RandomForest,0.978538,0.379747,0.016949,0.032450,0.823238,0.132533,79,False
4,LogisticRegression,0.978598,0.305556,0.006215,0.012182,0.815448,0.099958,36,False
5,KNN,0.978046,0.187500,0.010169,0.019293,0.587423,0.035812,96,False
6,DecisionTree,0.953333,0.086583,0.125424,0.102446,0.548558,0.029467,2564,False



=== TEST CONFUSION MATRICES ===

LogisticRegression
[[81561    25]
 [ 1759    11]]

DecisionTree
[[79244  2342]
 [ 1548   222]]

RandomForest
[[81537    49]
 [ 1740    30]]

GradientBoosting
[[81551    35]
 [ 1741    29]]

KNN
[[81508    78]
 [ 1752    18]]

XGBoost
[[81519    67]
 [ 1698    72]]

LightGBM
[[81469   117]
 [ 1698    72]]


### **Cost-Sensitive Model Training and Evaluation (Weighted Learning)**

In [59]:
neg_count = (y_train_raw == 0).sum()
pos_count = (y_train_raw == 1).sum()
scale_pos_weight = neg_count / pos_count

print("Train class counts:")
print(f"Negative (0): {neg_count:,}")
print(f"Positive (1): {pos_count:,}")
print(f"scale_pos_weight: {scale_pos_weight:.6f}")

models_weighted = {
    "LogisticRegression_weighted": {
        "model": LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "DecisionTree_weighted": {
        "model": DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "RandomForest_weighted": {
        "model": RandomForestClassifier(
            class_weight="balanced",
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
}

if xgb_available:
    models_weighted["XGBoost_weighted"] = {
        "model": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1
        ),
        "X_train": X_train_raw,
        "y_train": y_train_raw,
        "X_test": X_test_raw,
        "y_test": y_test_raw,
        "native_missing": True,
    }

if lgbm_available:
    models_weighted["LightGBM_weighted"] = {
        "model": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        "X_train": X_train_raw,
        "y_train": y_train_raw,
        "X_test": X_test_raw,
        "y_test": y_test_raw,
        "native_missing": True,
    }


weighted_results_df, weighted_conf_matrices = run_train_test_experiment(models_weighted)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("\n=== FINAL TEST RESULTS ===")
display(
    weighted_results_df[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "native_missing",
        ]
    ]
)

print("\n=== TEST CONFUSION MATRICES ===")
for model_name, cm in weighted_conf_matrices.items():
    print(f"\n{model_name}")
    print(cm)

Train class counts:
Negative (0): 372,201
Positive (1): 8,260
scale_pos_weight: 45.060654

Training LogisticRegression_weighted ...

Training DecisionTree_weighted ...

Training RandomForest_weighted ...

Training XGBoost_weighted ...

Training LightGBM_weighted ...

=== FINAL TEST RESULTS ===


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,XGBoost_weighted,0.812203,0.076914,0.712994,0.138849,0.858934,0.158109,16408,True
1,LightGBM_weighted,0.801070,0.072796,0.712994,0.132105,0.855593,0.150478,17336,True
2,RandomForest_weighted,0.978478,0.337838,0.014124,0.027115,0.830675,0.130430,74,False
3,LogisticRegression_weighted,0.729558,0.058528,0.777966,0.108867,0.831492,0.098531,23527,False
4,DecisionTree_weighted,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False



=== TEST CONFUSION MATRICES ===

LogisticRegression_weighted
[[59436 22150]
 [  393  1377]]

DecisionTree_weighted
[[79874  1712]
 [ 1582   188]]

RandomForest_weighted
[[81537    49]
 [ 1745    25]]

XGBoost_weighted
[[66440 15146]
 [  508  1262]]

LightGBM_weighted
[[65512 16074]
 [  508  1262]]


### **Threshold Tuning and Decision Boundary Optimization**

In [60]:
models_for_threshold = {
    "RandomForest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(
            random_state=42
        ),
        "X_train": X_train_imputed,
        "y_train": y_train_imputed,
        "X_test": X_test_imputed,
        "y_test": y_test_imputed,
        "native_missing": False,
    },
}

if xgb_available:
    models_for_threshold["XGBoost"] = {
        "model": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),
        "X_train": X_train_raw,
        "y_train": y_train_raw,
        "X_test": X_test_raw,
        "y_test": y_test_raw,
        "native_missing": True,
    }

if lgbm_available:
    models_for_threshold["LightGBM"] = {
        "model": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        "X_train": X_train_raw,
        "y_train": y_train_raw,
        "X_test": X_test_raw,
        "y_test": y_test_raw,
        "native_missing": True,
    }


thresholds = [
    0.01, 0.02, 0.03, 0.05, 0.07,
    0.10, 0.15, 0.20, 0.25, 0.30,
    0.40, 0.50
]

all_results = []
per_model_results = {}

for model_name, config in models_for_threshold.items():
    print(f"\nTraining {model_name} ...")

    model = config["model"]
    X_train = config["X_train"]
    y_train = config["y_train"]
    X_test = config["X_test"]
    y_test = config["y_test"]
    native_missing = config["native_missing"]

    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)

    model_rows = []

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)

        row = {
            "model": model_name,
            "threshold": t,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "predicted_positives": int((y_pred == 1).sum()),
            "native_missing": native_missing,
        }

        model_rows.append(row)
        all_results.append(row)

    model_df = pd.DataFrame(model_rows).sort_values(
        by=["f1", "recall", "precision"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    per_model_results[model_name] = model_df

results_df = pd.DataFrame(all_results)

best_per_model = (
    results_df.sort_values(
        by=["model", "f1", "recall", "precision"],
        ascending=[True, False, False, False]
    )
    .groupby("model", as_index=False)
    .first()
)

best_per_model = best_per_model.sort_values(
    by=["pr_auc", "recall", "f1"],
    ascending=[False, False, False]
).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("\n=== BEST THRESHOLD PER MODEL ===")
display(
    best_per_model[
        [
            "model",
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "native_missing",
        ]
    ]
)

print("\n=== THRESHOLD RESULTS BY MODEL ===")
ordered_models = best_per_model["model"].tolist()

for model_name in ordered_models:
    print(f"\n--- {model_name} ---")
    model_table = per_model_results[model_name].copy()
    model_table = model_table[
        [
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "native_missing",
        ]
    ]
    display(model_table)


Training RandomForest ...

Training GradientBoosting ...

Training XGBoost ...

Training LightGBM ...

=== BEST THRESHOLD PER MODEL ===


,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,XGBoost,0.150000,0.966697,0.228695,0.239548,0.233996,0.867742,0.174725,1854,True
1,LightGBM,0.100000,0.950789,0.166285,0.328249,0.220745,0.861630,0.154859,3494,True
2,GradientBoosting,0.100000,0.958215,0.182895,0.279096,0.220980,0.857717,0.145876,2701,False
3,RandomForest,0.150000,0.963266,0.190019,0.223729,0.205501,0.823238,0.132533,2084,False



=== THRESHOLD RESULTS BY MODEL ===

--- XGBoost ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.150000,0.966697,0.228695,0.239548,0.233996,0.867742,0.174725,1854,True
1,0.100000,0.950861,0.170538,0.340113,0.227170,0.867742,0.174725,3530,True
2,0.200000,0.973295,0.292350,0.181356,0.223849,0.867742,0.174725,1098,True
3,0.070000,0.924289,0.130633,0.453672,0.202855,0.867742,0.174725,6147,True
4,0.250000,0.975983,0.329912,0.127119,0.183524,0.867742,0.174725,682,True
5,0.050000,0.890986,0.108591,0.573446,0.182603,0.867742,0.174725,9347,True
6,0.300000,0.977314,0.369892,0.097175,0.153915,0.867742,0.174725,465,True
7,0.030000,0.819353,0.080979,0.725424,0.145694,0.867742,0.174725,15856,True
8,0.020000,0.749616,0.066182,0.823164,0.122514,0.867742,0.174725,22015,True
9,0.400000,0.978382,0.433333,0.058757,0.103483,0.867742,0.174725,240,True



--- LightGBM ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.100000,0.950789,0.166285,0.328249,0.220745,0.861630,0.154859,3494,True
1,0.150000,0.965569,0.211740,0.228249,0.219685,0.861630,0.154859,1908,True
2,0.200000,0.972216,0.262195,0.170056,0.206306,0.861630,0.154859,1148,True
3,0.070000,0.926112,0.129620,0.433898,0.199610,0.861630,0.154859,5925,True
4,0.050000,0.892461,0.106197,0.548023,0.177916,0.861630,0.154859,9134,True
5,0.250000,0.976067,0.326656,0.119774,0.175279,0.861630,0.154859,649,True
6,0.300000,0.977182,0.358369,0.094350,0.149374,0.861630,0.154859,466,True
7,0.030000,0.820805,0.080589,0.714689,0.144845,0.861630,0.154859,15697,True
8,0.020000,0.749112,0.065110,0.809605,0.120527,0.861630,0.154859,22009,True
9,0.400000,0.978034,0.390681,0.061582,0.106393,0.861630,0.154859,279,True



--- GradientBoosting ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.100000,0.958215,0.182895,0.279096,0.220980,0.857717,0.145876,2701,False
1,0.150000,0.970200,0.236726,0.181356,0.205374,0.857717,0.145876,1356,False
2,0.070000,0.934798,0.137631,0.393220,0.203896,0.857717,0.145876,5057,False
3,0.050000,0.898999,0.110486,0.532768,0.183018,0.857717,0.145876,8535,False
4,0.200000,0.974735,0.281250,0.122034,0.170213,0.857717,0.145876,768,False
5,0.030000,0.813787,0.077641,0.714124,0.140055,0.857717,0.145876,16280,False
6,0.250000,0.976342,0.302734,0.087571,0.135846,0.857717,0.145876,512,False
7,0.020000,0.724579,0.060743,0.827684,0.113180,0.857717,0.145876,24118,False
8,0.300000,0.977650,0.352381,0.062712,0.106475,0.857717,0.145876,315,False
9,0.010000,0.542912,0.042212,0.946328,0.080818,0.857717,0.145876,39681,False



--- RandomForest ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.150000,0.963266,0.190019,0.223729,0.205501,0.823238,0.132533,2084,False
1,0.100000,0.937221,0.136163,0.366102,0.198499,0.823238,0.132533,4759,False
2,0.200000,0.972635,0.259189,0.155367,0.194278,0.823238,0.132533,1061,False
3,0.070000,0.892041,0.097808,0.496610,0.163428,0.823238,0.132533,8987,False
4,0.250000,0.975599,0.296923,0.109040,0.159504,0.823238,0.132533,650,False
5,0.050000,0.832310,0.076229,0.620339,0.135773,0.823238,0.132533,14404,False
6,0.300000,0.977134,0.335749,0.078531,0.127289,0.823238,0.132533,414,False
7,0.030000,0.721148,0.056689,0.775706,0.105656,0.823238,0.132533,24220,False
8,0.020000,0.627129,0.046368,0.846328,0.087919,0.823238,0.132533,32307,False
9,0.400000,0.978250,0.387435,0.041808,0.075472,0.823238,0.132533,191,False


### **Weighted Models + Threshold Optimization (Joint Evaluation)**

In [61]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

all_results = []
per_model_results = {}

for model_name, config in models_weighted.items():
    print(f"\nTraining {model_name} ...")

    model = config["model"]
    X_train = config["X_train"]
    y_train = config["y_train"]
    X_test = config["X_test"]
    y_test = config["y_test"]
    native_missing = config["native_missing"]

    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)

    model_rows = []

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)

        row = {
            "model": model_name,
            "threshold": t,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "predicted_positives": int((y_pred == 1).sum()),
            "native_missing": native_missing,
        }

        model_rows.append(row)
        all_results.append(row)

    model_df = pd.DataFrame(model_rows).sort_values(
        by=["f1", "recall", "precision"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    per_model_results[model_name] = model_df

results_df = pd.DataFrame(all_results)

best_per_model = (
    results_df.sort_values(
        by=["model", "f1", "recall", "precision"],
        ascending=[True, False, False, False]
    )
    .groupby("model", as_index=False)
    .first()
)

best_per_model = best_per_model.sort_values(
    by=["pr_auc", "recall", "f1"],
    ascending=[False, False, False]
).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("\n=== BEST THRESHOLD PER WEIGHTED MODEL (BEST TO WORST) ===")
display(
    best_per_model[
        [
            "model",
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "native_missing",
        ]
    ]
)

print("\n=== THRESHOLD RESULTS BY WEIGHTED MODEL ===")

ordered_models = best_per_model["model"].tolist()

for model_name in ordered_models:
    print(f"\n--- {model_name} ---")
    model_table = per_model_results[model_name].copy()
    model_table = model_table[
        [
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "native_missing",
        ]
    ]
    display(model_table)


Training LogisticRegression_weighted ...

Training DecisionTree_weighted ...

Training RandomForest_weighted ...

Training XGBoost_weighted ...

Training LightGBM_weighted ...

=== BEST THRESHOLD PER WEIGHTED MODEL (BEST TO WORST) ===


,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,XGBoost_weighted,0.800000,0.948246,0.164380,0.351977,0.224101,0.858934,0.158109,3790,True
1,LightGBM_weighted,0.800000,0.942524,0.152998,0.376271,0.217540,0.855593,0.150478,4353,True
2,RandomForest_weighted,0.100000,0.953729,0.160650,0.279096,0.203922,0.830675,0.130430,3075,False
3,LogisticRegression_weighted,0.800000,0.937485,0.117582,0.298870,0.168767,0.831492,0.098531,4499,False
4,DecisionTree_weighted,0.100000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False



=== THRESHOLD RESULTS BY WEIGHTED MODEL ===

--- XGBoost_weighted ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.800000,0.948246,0.164380,0.351977,0.224101,0.858934,0.158109,3790,True
1,0.900000,0.972659,0.268426,0.166667,0.205647,0.858934,0.158109,1099,True
2,0.700000,0.910408,0.119219,0.503955,0.192823,0.858934,0.158109,7482,True
3,0.600000,0.864677,0.094491,0.625989,0.164197,0.858934,0.158109,11726,True
4,0.500000,0.812203,0.076914,0.712994,0.138849,0.858934,0.158109,16408,True
5,0.400000,0.751907,0.064246,0.787571,0.118800,0.858934,0.158109,21698,True
6,0.300000,0.679891,0.054377,0.858757,0.102278,0.858934,0.158109,27953,True
7,0.200000,0.588884,0.045685,0.923164,0.087061,0.858934,0.158109,35767,True
8,0.100000,0.462786,0.036929,0.968927,0.071147,0.858934,0.158109,46440,True



--- LightGBM_weighted ---


...

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.100000,0.953729,0.160650,0.279096,0.203922,0.830675,0.130430,3075,False
1,0.200000,0.974231,0.271186,0.126554,0.172573,0.830675,0.130430,826,False
2,0.300000,0.977410,0.339031,0.067232,0.112211,0.830675,0.130430,351,False
3,0.400000,0.978154,0.345455,0.032203,0.058915,0.830675,0.130430,165,False
4,0.500000,0.978478,0.353659,0.016384,0.031317,0.830675,0.130430,82,False
5,0.600000,0.978622,0.300000,0.005085,0.010000,0.830675,0.130430,30,False
6,0.700000,0.978754,0.428571,0.001695,0.003376,0.830675,0.130430,7,False
7,0.800000,0.978766,0.000000,0.000000,0.000000,0.830675,0.130430,0,False
8,0.900000,0.978766,0.000000,0.000000,0.000000,0.830675,0.130430,0,False



--- LogisticRegression_weighted ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.800000,0.937485,0.117582,0.298870,0.168767,0.831492,0.098531,4499,False
1,0.700000,0.885239,0.090632,0.487571,0.152852,0.831492,0.098531,9522,False
2,0.900000,0.968017,0.168639,0.128814,0.146060,0.831492,0.098531,1352,False
3,0.600000,0.814075,0.072390,0.656497,0.130401,0.831492,0.098531,16052,False
4,0.500000,0.729558,0.058528,0.777966,0.108867,0.831492,0.098531,23527,False
5,0.400000,0.630321,0.048135,0.874011,0.091244,0.831492,0.098531,32139,False
6,0.300000,0.509933,0.039368,0.943503,0.075583,0.831492,0.098531,42420,False
7,0.200000,0.355019,0.031214,0.977966,0.060498,0.831492,0.098531,55455,False
8,0.100000,0.176376,0.025012,0.994915,0.048797,0.831492,0.098531,70406,False



--- DecisionTree_weighted ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,native_missing
0,0.100000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
1,0.200000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
2,0.300000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
3,0.400000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
4,0.500000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
5,0.600000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
6,0.700000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
7,0.800000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False
8,0.900000,0.960483,0.098947,0.106215,0.102452,0.542615,0.029488,1900,False


### **SMOTETomek Balancing**

In [6]:
# =========================
# SMOTETOMEK BALANCING EXPERIMENT
# Uses imputed splits for all models
# Resampling is applied ONLY on X_train
# =========================

# -------------------------
# 1) Define models for SMOTETomek experiment
# -------------------------
models_smotetomek = {
    "LogisticRegression_SMOTETomek": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "DecisionTree_SMOTETomek": DecisionTreeClassifier(
        random_state=42
    ),
    "RandomForest_SMOTETomek": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting_SMOTETomek": GradientBoostingClassifier(
        random_state=42
    ),
    "KNN_SMOTETomek": KNeighborsClassifier(
        n_neighbors=5
    ),
}

if xgb_available:
    models_smotetomek["XGBoost_SMOTETomek"] = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

if lgbm_available:
    models_smotetomek["LightGBM_SMOTETomek"] = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

# -------------------------
# 2) Apply SMOTETomek ONLY on training data
# -------------------------
smote_tomek = SMOTETomek(random_state=42)

X_train_smotetomek, y_train_smotetomek = smote_tomek.fit_resample(
    X_train_imputed,
    y_train_imputed
)

print("Original train shape:", X_train_imputed.shape)
print("Resampled train shape:", X_train_smotetomek.shape)

print("\nOriginal train target distribution:")
print(y_train_imputed.value_counts())

print("\nResampled train target distribution:")
print(pd.Series(y_train_smotetomek).value_counts())

# -------------------------
# 3) Train + evaluate all models
# -------------------------
all_results = []
conf_matrices = {}

for model_name, model in models_smotetomek.items():
    print(f"\nTraining {model_name} ...")

    model.fit(X_train_smotetomek, y_train_smotetomek)

    y_pred = model.predict(X_test_imputed)
    y_score = get_scores(model, X_test_imputed)

    result = compute_metrics(model_name, y_test_imputed, y_pred, y_score)
    result["balancing_method"] = "SMOTETomek"

    all_results.append(result)
    conf_matrices[model_name] = confusion_matrix(y_test_imputed, y_pred)

# -------------------------
# 4) Final results table
# -------------------------
smotetomek_results_df = pd.DataFrame(all_results).sort_values(
    by=["pr_auc", "recall", "f1"],
    ascending=False
).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("\n=== SMOTETOMEK RESULTS ===")
display(
    smotetomek_results_df[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "predicted_positives",
            "balancing_method",
        ]
    ]
)

# -------------------------
# 5) Confusion matrices
# -------------------------
print("\n=== SMOTETOMEK CONFUSION MATRICES ===")
for model_name, cm in conf_matrices.items():
    print(f"\n{model_name}")
    print(cm)

Original train shape: (380461, 49)
Resampled train shape: (744228, 49)

Original train target distribution:
target
0    372201
1      8260
Name: count, dtype: int64

Resampled train target distribution:
target
0    372114
1    372114
Name: count, dtype: int64

Training LogisticRegression_SMOTETomek ...

Training DecisionTree_SMOTETomek ...

Training RandomForest_SMOTETomek ...

Training GradientBoosting_SMOTETomek ...

Training KNN_SMOTETomek ...

Training XGBoost_SMOTETomek ...

Training LightGBM_SMOTETomek ...

=== SMOTETOMEK RESULTS ===


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,balancing_method
0,LightGBM_SMOTETomek,0.957580,0.135124,0.184746,0.156086,0.808068,0.089111,2420,SMOTETomek
1,XGBoost_SMOTETomek,0.937257,0.107175,0.266667,0.152899,0.796380,0.087651,4404,SMOTETomek
2,RandomForest_SMOTETomek,0.967105,0.145773,0.112994,0.127307,0.809442,0.086666,1372,SMOTETomek
3,GradientBoosting_SMOTETomek,0.889018,0.078345,0.392655,0.130627,0.781050,0.081999,8871,SMOTETomek
4,LogisticRegression_SMOTETomek,0.797591,0.053776,0.514124,0.097368,0.738755,0.059200,16922,SMOTETomek
5,KNN_SMOTETomek,0.828423,0.047908,0.375141,0.084965,0.642793,0.036279,13860,SMOTETomek
6,DecisionTree_SMOTETomek,0.929195,0.066695,0.179661,0.097277,0.563021,0.029416,4768,SMOTETomek



=== SMOTETOMEK CONFUSION MATRICES ===

LogisticRegression_SMOTETomek
[[65574 16012]
 [  860   910]]

DecisionTree_SMOTETomek
[[77136  4450]
 [ 1452   318]]

RandomForest_SMOTETomek
[[80414  1172]
 [ 1570   200]]

GradientBoosting_SMOTETomek
[[73410  8176]
 [ 1075   695]]

KNN_SMOTETomek
[[68390 13196]
 [ 1106   664]]

XGBoost_SMOTETomek
[[77654  3932]
 [ 1298   472]]

LightGBM_SMOTETomek
[[79493  2093]
 [ 1443   327]]


### **XGBoost Tuning**

In [7]:
# =========================
# XGBOOST RANDOM SEARCH + CALIBRATION + THRESHOLD SEARCH
# Uses RAW splits
# Updated pipeline:
# - Group-safe internal fit/calibration/validation split
# - Randomized hyperparameter search (instead of full grid)
# - scale_pos_weight for imbalance handling
# - Best base model selected by calibration PR-AUC
# - Calibration fit on calibration split and compared on validation
# - Threshold search on validation only
# - Final evaluation on TEST and DEPLOYMENT
# =========================

# -------------------------
# 0) Imports
# -------------------------
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, ParameterSampler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from xgboost import XGBClassifier

# -------------------------
# 1) Group-safe internal split:
#    train -> fit + calibration + validation
# -------------------------
TRAIN_FULL_PATH = "../data/processed/splits/train_full.csv"
train_full_raw = pd.read_csv(TRAIN_FULL_PATH)
feature_cols = X_train_raw.columns.tolist()

missing_required_cols = {"subject_id", "target"}.difference(train_full_raw.columns)
if missing_required_cols:
    raise ValueError(f"Missing required columns in train_full.csv: {sorted(missing_required_cols)}")

outer_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
fit_idx, holdout_idx = next(
    outer_splitter.split(train_full_raw, groups=train_full_raw["subject_id"])
)

fit_df = train_full_raw.iloc[fit_idx].reset_index(drop=True)
holdout_df = train_full_raw.iloc[holdout_idx].reset_index(drop=True)

inner_splitter = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
cal_idx, val_idx = next(
    inner_splitter.split(holdout_df, groups=holdout_df["subject_id"])
)

cal_df = holdout_df.iloc[cal_idx].reset_index(drop=True)
val_df = holdout_df.iloc[val_idx].reset_index(drop=True)

X_fit = fit_df[feature_cols].copy()
y_fit = fit_df["target"].copy()
X_cal = cal_df[feature_cols].copy()
y_cal = cal_df["target"].copy()
X_val = val_df[feature_cols].copy()
y_val = val_df["target"].copy()

fit_subjects = set(fit_df["subject_id"])
cal_subjects = set(cal_df["subject_id"])
val_subjects = set(val_df["subject_id"])

print("Group-safe internal split shapes:")
print("X_fit:", X_fit.shape)
print("X_cal:", X_cal.shape)
print("X_val:", X_val.shape)
print("X_test_raw:", X_test_raw.shape)
print("X_deployment_raw:", X_deployment_raw.shape)

print("\nSubject overlap checks:")
print("fit/cal overlap:", len(fit_subjects & cal_subjects))
print("fit/val overlap:", len(fit_subjects & val_subjects))
print("cal/val overlap:", len(cal_subjects & val_subjects))

# -------------------------
# 2) Imbalance handling
# -------------------------
n_negative = (y_fit == 0).sum()
n_positive = (y_fit == 1).sum()
scale_pos_weight_value = n_negative / n_positive

print("\nClass balance on X_fit:")
print("Negatives:", n_negative)
print("Positives:", n_positive)
print("scale_pos_weight:", round(scale_pos_weight_value, 4))

# -------------------------
# 3) Randomized search space
# -------------------------
param_distributions = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.5, 1, 2],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10],
    "max_delta_step": [0, 1, 3, 5],
}

n_iter_search = 30
random_state = 42

sampled_params = list(
    ParameterSampler(
        param_distributions=param_distributions,
        n_iter=n_iter_search,
        random_state=random_state
    )
)

print(f"\nNumber of sampled parameter combinations: {len(sampled_params)}")

# -------------------------
# 4) Helper functions
# -------------------------
def evaluate_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "predicted_positives": int((y_pred == 1).sum()),
        "y_pred": y_pred,
    }

def fit_calibrated_model(base_model, method, X_cal, y_cal):
    """
    method:
        - "none"     -> no calibration, return base_model
        - "sigmoid"  -> Platt scaling using calibration split
        - "isotonic" -> isotonic calibration using calibration split
    """
    if method == "none":
        return base_model

    calibrated_model = CalibratedClassifierCV(
        estimator=FrozenEstimator(base_model),
        method=method,
    )

    calibrated_model.fit(X_cal, y_cal)
    return calibrated_model

# -------------------------
# 5) Randomized search for best BASE model
#    Selection priority:
#    calibration PR-AUC -> calibration ROC-AUC
# -------------------------
base_search_results = []

for i, params in enumerate(sampled_params, start=1):
    print(f"\n[{i}/{len(sampled_params)}] Testing params: {params}")

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight_value,
        **params
    )

    model.fit(X_fit, y_fit)

    y_cal_proba = model.predict_proba(X_cal)[:, 1]
    cal_pr_auc = average_precision_score(y_cal, y_cal_proba)
    cal_roc_auc = roc_auc_score(y_cal, y_cal_proba)

    base_search_results.append({
        "params": params,
        "cal_pr_auc": float(cal_pr_auc),
        "cal_roc_auc": float(cal_roc_auc),
    })

base_results_df = pd.DataFrame(base_search_results).sort_values(
    by=["cal_pr_auc", "cal_roc_auc"],
    ascending=False
).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("\n=== BASE MODEL RANDOM SEARCH RESULTS (BEST TO WORST) ===")
display(base_results_df.head(15))

best_base_row = base_results_df.iloc[0]
best_params = best_base_row["params"]

print("\n=== BEST BASE MODEL CONFIGURATION ===")
print("Best params:", best_params)
print("Best calibration PR-AUC:", best_base_row["cal_pr_auc"])
print("Best calibration ROC-AUC:", best_base_row["cal_roc_auc"])

# -------------------------
# 6) Fit best base model on fit set
# -------------------------
best_base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight_value,
    **best_params
)

best_base_model.fit(X_fit, y_fit)

# -------------------------
# 7) Compare calibration methods on validation
#    Selection priority:
#    validation PR-AUC -> validation ROC-AUC
#    Calibration models are fit on the calibration split only.
# -------------------------
calibration_methods = ["none", "sigmoid", "isotonic"]
calibration_results = []

for calib_method in calibration_methods:
    print(f"\nTesting calibration method: {calib_method}")

    calibrated_model = fit_calibrated_model(
        base_model=best_base_model,
        method=calib_method,
        X_cal=X_cal,
        y_cal=y_cal,
    )

    y_val_proba = calibrated_model.predict_proba(X_val)[:, 1]
    val_pr_auc = average_precision_score(y_val, y_val_proba)
    val_roc_auc = roc_auc_score(y_val, y_val_proba)

    calibration_results.append({
        "calibration_method": calib_method,
        "val_pr_auc": float(val_pr_auc),
        "val_roc_auc": float(val_roc_auc),
    })

calibration_results_df = pd.DataFrame(calibration_results).sort_values(
    by=["val_pr_auc", "val_roc_auc"],
    ascending=False
).reset_index(drop=True)

print("\n=== CALIBRATION COMPARISON ===")
display(calibration_results_df)

best_calibration_method = calibration_results_df.iloc[0]["calibration_method"]

print("\n=== BEST CALIBRATION METHOD ===")
print("Best calibration method:", best_calibration_method)

# -------------------------
# 8) Refit best calibrated model
# -------------------------
final_base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight_value,
    **best_params
)

final_base_model.fit(X_fit, y_fit)

final_model = fit_calibrated_model(
    base_model=final_base_model,
    method=best_calibration_method,
    X_cal=X_cal,
    y_cal=y_cal,
)

# -------------------------
# 9) Threshold search on validation only
#    Selection priority:
#    F1 -> Recall -> Precision
# -------------------------
thresholds = np.round(np.arange(0.01, 0.991, 0.01), 2)

y_val_proba_final = final_model.predict_proba(X_val)[:, 1]

threshold_rows = []
for t in thresholds:
    row = evaluate_threshold(y_val, y_val_proba_final, t)
    row["threshold"] = t
    threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows).sort_values(
    by=["f1", "recall", "precision"],
    ascending=False
).reset_index(drop=True)

best_threshold_row = threshold_df.iloc[0]
best_threshold = float(best_threshold_row["threshold"])

print("\n=== BEST THRESHOLD ON VALIDATION ===")
display(threshold_df.head(15))

print("\nSelected threshold:", best_threshold)

# -------------------------
# 10) Final TEST evaluation
# -------------------------
y_test_proba_best = final_model.predict_proba(X_test_raw)[:, 1]
test_eval = evaluate_threshold(y_test_raw, y_test_proba_best, best_threshold)

xgb_tuned_test_results = pd.DataFrame([{
    "model": "XGBoost_random_search_calibrated_thresholded",
    "accuracy": test_eval["accuracy"],
    "precision": test_eval["precision"],
    "recall": test_eval["recall"],
    "f1": test_eval["f1"],
    "roc_auc": roc_auc_score(y_test_raw, y_test_proba_best),
    "pr_auc": average_precision_score(y_test_raw, y_test_proba_best),
    "predicted_positives": test_eval["predicted_positives"],
    "threshold": best_threshold,
    "calibration_method": best_calibration_method,
    "scale_pos_weight": float(scale_pos_weight_value),
}])

xgb_tuned_test_cm = confusion_matrix(y_test_raw, test_eval["y_pred"])

print("\n=== FINAL TEST RESULTS ===")
display(xgb_tuned_test_results)

print("\n=== TEST CONFUSION MATRIX ===")
print(xgb_tuned_test_cm)

# -------------------------
# 11) Final DEPLOYMENT evaluation
# -------------------------
y_dep_proba_best = final_model.predict_proba(X_deployment_raw)[:, 1]
dep_eval = evaluate_threshold(y_deployment_raw, y_dep_proba_best, best_threshold)

xgb_tuned_deployment_results = pd.DataFrame([{
    "model": "XGBoost_random_search_calibrated_thresholded_deployment",
    "accuracy": dep_eval["accuracy"],
    "precision": dep_eval["precision"],
    "recall": dep_eval["recall"],
    "f1": dep_eval["f1"],
    "roc_auc": roc_auc_score(y_deployment_raw, y_dep_proba_best),
    "pr_auc": average_precision_score(y_deployment_raw, y_dep_proba_best),
    "predicted_positives": dep_eval["predicted_positives"],
    "threshold": best_threshold,
    "calibration_method": best_calibration_method,
    "scale_pos_weight": float(scale_pos_weight_value),
}])

xgb_tuned_deployment_cm = confusion_matrix(y_deployment_raw, dep_eval["y_pred"])

print("\n=== DEPLOYMENT RESULTS ===")
display(xgb_tuned_deployment_results)

print("\n=== DEPLOYMENT CONFUSION MATRIX ===")
print(xgb_tuned_deployment_cm)

Group-safe internal split shapes:
X_fit: (303600, 49)
X_cal: (38061, 49)
X_val: (38800, 49)
X_test_raw: (83356, 49)
X_deployment_raw: (82211, 49)

Subject overlap checks:
fit/cal overlap: 0
fit/val overlap: 0
cal/val overlap: 0

Class balance on X_fit:
Negatives: 296994
Positives: 6606
scale_pos_weight: 44.9582

Number of sampled parameter combinations: 30

[1/30] Testing params: {'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 10, 'max_delta_step': 0, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 0.9}

[2/30] Testing params: {'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 5, 'max_delta_step': 0, 'learning_rate': 0.01, 'gamma': 1, 'colsample_bytree': 1.0}

[3/30] Testing params: {'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 4, 'max_delta_step': 3, 'learning_rate': 0.05, 'gamma': 1, 'colsamp

,params,cal_pr_auc,cal_roc_auc
0,"{'subsample': 0.7, 'reg_lambda': 10, 'reg_alph...",0.177735,0.862576
1,"{'subsample': 1.0, 'reg_lambda': 5, 'reg_alpha...",0.176445,0.857743
2,"{'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha...",0.174740,0.861041
3,"{'subsample': 0.9, 'reg_lambda': 5, 'reg_alpha...",0.173854,0.857016
4,"{'subsample': 0.7, 'reg_lambda': 10, 'reg_alph...",0.172615,0.861656
5,"{'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha...",0.171993,0.859843
6,"{'subsample': 1.0, 'reg_lambda': 2, 'reg_alpha...",0.171837,0.855827
7,"{'subsample': 0.7, 'reg_lambda': 2, 'reg_alpha...",0.170829,0.858186
8,"{'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha...",0.170754,0.857211
9,"{'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha...",0.170679,0.857970



=== BEST BASE MODEL CONFIGURATION ===
Best params: {'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 5, 'max_delta_step': 3, 'learning_rate': 0.07, 'gamma': 0, 'colsample_bytree': 0.6}
Best calibration PR-AUC: 0.1777351511315659
Best calibration ROC-AUC: 0.8625764493518617

Testing calibration method: none

Testing calibration method: sigmoid

Testing calibration method: isotonic

=== CALIBRATION COMPARISON ===


,calibration_method,val_pr_auc,val_roc_auc
0,none,0.154539,0.857930
1,sigmoid,0.154539,0.857930
2,isotonic,0.143670,0.856944



=== BEST CALIBRATION METHOD ===
Best calibration method: none

=== BEST THRESHOLD ON VALIDATION ===


...


Selected threshold: 0.84

=== FINAL TEST RESULTS ===


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,threshold,calibration_method,scale_pos_weight
0,XGBoost_random_search_calibrated_thresholded,0.959295,0.193658,0.289831,0.232179,0.865479,0.164435,2649,0.840000,none,44.958220



=== TEST CONFUSION MATRIX ===
[[79450  2136]
 [ 1257   513]]

=== DEPLOYMENT RESULTS ===


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,predicted_positives,threshold,calibration_method,scale_pos_weight
0,XGBoost_random_search_calibrated_thresholded_d...,0.959507,0.194270,0.279503,0.229220,0.866552,0.164347,2548,0.840000,none,44.958220



=== DEPLOYMENT CONFUSION MATRIX ===
[[78387  2053]
 [ 1276   495]]


In [8]:
# -------------------------
# 12) Save model + artifacts
# -------------------------
ARTIFACTS_DIR = Path("../artifacts/xgboost")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, ARTIFACTS_DIR / "xgb_model.joblib")

pd.DataFrame({
    "feature_names": X_train_raw.columns
}).to_csv(ARTIFACTS_DIR / "feature_names.csv", index=False)

X_test_raw.to_csv(ARTIFACTS_DIR / "X_test_used.csv", index=False)
y_test_raw.to_csv(ARTIFACTS_DIR / "y_test_used.csv", index=False)

X_deployment_raw.to_csv(ARTIFACTS_DIR / "X_deployment_used.csv", index=False)
y_deployment_raw.to_csv(ARTIFACTS_DIR / "y_deployment_used.csv", index=False)

X_background = X_train_raw.sample(
    n=min(200, len(X_train_raw)),
    random_state=42
).reset_index(drop=True)
X_background.to_csv(ARTIFACTS_DIR / "X_background_for_shap.csv", index=False)

pd.DataFrame({
    "y_true": y_test_raw.values,
    "y_proba": y_test_proba_best,
    "y_pred": test_eval["y_pred"],
}).to_csv(ARTIFACTS_DIR / "test_predictions.csv", index=False)

pd.DataFrame({
    "y_true": y_deployment_raw.values,
    "y_proba": y_dep_proba_best,
    "y_pred": dep_eval["y_pred"],
}).to_csv(ARTIFACTS_DIR / "deployment_predictions.csv", index=False)

base_results_df.to_csv(ARTIFACTS_DIR / "base_search_results.csv", index=False)
calibration_results_df.to_csv(ARTIFACTS_DIR / "calibration_results.csv", index=False)
threshold_df.to_csv(ARTIFACTS_DIR / "validation_threshold_results.csv", index=False)

metadata = {
    "model_name": "XGBoost_random_search_calibrated_thresholded",
    "best_params": best_params,
    "best_calibration_method": best_calibration_method,
    "best_threshold": best_threshold,
    "scale_pos_weight": float(scale_pos_weight_value),
    "base_model_selection_metric_order": ["val_pr_auc", "val_roc_auc"],
    "calibration_selection_metric_order": ["val_pr_auc", "val_roc_auc"],
    "threshold_selection_metric_order": ["f1", "recall", "precision"],
    "n_iter_search": n_iter_search,
    "artifacts_saved_in": str(ARTIFACTS_DIR),
}

with open(ARTIFACTS_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("\nSaved artifacts to:", ARTIFACTS_DIR.resolve())


Saved artifacts to: /Users/sarrachouk/Desktop/ehr-mortality-prediction/artifacts/xgboost
